## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


improve reproducibility by using standardized nf-core workflows, fixing pipeline and reference-data versions, and running tools in containers with defined software versions. 
also record the input samplesheets, parameters and commands, and keep the analysis code under version control so that the same analysis can be repeated.

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

The next step is to perform quality control and process the raw FASTQ files to generate a gene count matrix.

Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

We would use nf-core/rnaseq, which performs read QC, trimming, alignment and quantification, and summarizes quality-control results in a MultiQC report.

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [5]:
# post here the command you used to run nf-core/rnaseq
import pandas as pd

metadata = pd.read_csv("selected_runs_metadata.csv")

metadata["sample"] = (
    metadata["Run"] + "_" +
    metadata["condition"] + "_" +
    metadata["genotype"]
)

metadata[["sample", "Run", "condition", "genotype"]]

,sample,Run,condition,genotype
0,SRR23195516_oxy_SNI,SRR23195516,oxy,SNI
1,SRR23195511_oxy_Sham,SRR23195511,oxy,Sham


In [7]:
from pathlib import Path
fastq_dir = Path(
    "/home/zwc/computational-workflows-2026/day_02/SRA_runs/fastq"
    
)
for file in sorted(fastq_dir.glob("*.fastq.gz")):
    print(file.name)

SRX19144486_SRR23195516_1.fastq.gz
SRX19144486_SRR23195516_2.fastq.gz
SRX19144488_SRR23195511_1.fastq.gz
SRX19144488_SRR23195511_2.fastq.gz


In [8]:
samplesheet = pd.DataFrame({
    "sample": [
        "SRR23195516_oxy_SNI",
        "SRR23195511_oxy_Sham"
    ],
    "fastq_1": [
        str(fastq_dir / "SRX19144486_SRR23195516_1.fastq.gz"),
        str(fastq_dir / "SRX19144488_SRR23195511_1.fastq.gz")
    ],
    "fastq_2": [
        str(fastq_dir / "SRX19144486_SRR23195516_2.fastq.gz"),
        str(fastq_dir / "SRX19144488_SRR23195511_2.fastq.gz")
    ],
    "strandedness": ["auto", "auto"]
})

samplesheet

,sample,fastq_1,fastq_2,strandedness
0,SRR23195516_oxy_SNI,/home/zwc/computational-workflows-2026/day_02/...,/home/zwc/computational-workflows-2026/day_02/...,auto
1,SRR23195511_oxy_Sham,/home/zwc/computational-workflows-2026/day_02/...,/home/zwc/computational-workflows-2026/day_02/...,auto


In [9]:
for column in ["fastq_1", "fastq_2"]:
    for filename in samplesheet[column]:
        assert Path(filename).is_file(), f"cant find file：{filename}"

output = Path(
    "/home/zwc/computational-workflows-2026/day_02/samplesheet.csv"
)
samplesheet.to_csv(output, index=False)

Explain all the parameters you set and why you set them in this way.



nextflow run nf-core/rnaseq \
  -r 3.27.0 \
  -profile docker \
  -c laptop.config \
  --input samplesheet.csv \
  --outdir rnaseq_results \
  --genome GRCm38 \ #Automatically use the corresponding mouse genome FASTA and GTF files. -> takes too long
  --aligner hisat2 \ #consistent with the alignment software used in the paper and is better suited to 12 GB of RAM than STAR
  --pseudo_aligner salmon \ #Generate the expression levels and count matrix required for downstream analysis
  -resume


mkdir -p reference_chr19_v102

curl -fL --retry 3 -o reference_chr19_v102/Mus_musculus.GRCm38.dna.chromosome.19.fa.gz \
  https://ftp.ensembl.org/pub/release-102/fasta/mus_musculus/dna/Mus_musculus.GRCm38.dna.chromosome.19.fa.gz

curl -fL --retry 3 -o reference_chr19_v102/Mus_musculus.GRCm38.102.gtf.gz \
  https://ftp.ensembl.org/pub/release-102/gtf/mus_musculus/Mus_musculus.GRCm38.102.gtf.gz


nextflow run nf-core/rnaseq \
  -r 3.27.0 \
  -profile docker \
  -c laptop.config \
  -params-file params_chr19.json \
  --input samplesheet.csv \
  --outdir rnaseq_chr19_v102_results \
  --fasta reference_chr19_v102/Mus_musculus.GRCm38.dna.chromosome.19.fa.gz \
  --gtf reference_chr19_v102/Mus_musculus.GRCm38.102.chr19.gtf.gz \
  --aligner hisat2 \
  --pseudo_aligner salmon \
  -resume

## Browsing the results

How did the pipeline perform?

The pipeline completed successfully in 1 hour 23 minutes (56 tasks succeeded, 6 were cached). I first verified and replaced FASTQ files whose MD5 checksums did not match ENA’s records, then reran nf-core/rnaseq with -resume. To fit my laptop’s memory, I used a reference containing only mouse chromosome 19, HISAT2 for alignment and Salmon for quantification. The pipeline produced a gene count table containing both samples and a MultiQC report. Alignment rates were low (about 1.9% and 2.6%), which is expected when reads from the whole genome are aligned to chromosome 19 alone. With only one sample per group, these results cannot support a reliable differential expression analysis.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

The pipeline checked FASTQ format, read quality, adapter trimming, strand direction and alignment, then combined the results in MultiQC. Both samples produced gene counts. I am reasonably happy that the pipeline worked, but I cannot fully judge the data quality: alignment was only about 2%, mainly because I used chromosome 19 rather than the whole mouse genome.
- rRNA: A high proportion of ribosomal RNA would mean that many reads are not useful for measuring gene expression. I could not reliably assess the rRNA proportion from this run.
- Duplication: Repeated reads can result from PCR amplification or very highly expressed genes. The Sham sample shows 21.1% duplication; this number alone does not prove a problem.
- GC content: The GC distribution should be reasonably smooth. An unusual shape could suggest contamination or library-preparation bias. In my run, the FastQC GC-content distributions peak around 46–49% and are similar between paired reads. There are some smaller secondary peaks, so GC content alone cannot rule out contamination, but I do not see a major GC mismatch between R1 and R2.

Poor RNA extraction or degradation, incomplete rRNA removal, too much PCR amplification, contamination, and sequencing or adapter problems could reduce quality. Using a chromosome 19-only reference also lowers the measured alignment rate, even if the reads themselves are good.

Would you exclude any samples? If yes, which and why?

I would not exclude either sample at this stage. Both passed FASTQ validation and produced gene counts. The low alignment rates are expected because I used only chromosome 19 as the reference, and the observed duplication and GC results do not by themselves justify exclusion. I would assess both samples again with a full-genome reference before making a final quality decision.

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

The scientists want to understand how nerve injury (SNI) and oxycodone exposure and withdrawal affect gene expression in brain regions involved in pain and reward, and whether oxycodone has a different effect in SNI mice than in Sham mice.

To continue, I would analyze all samples from the four groups, using the full mouse genome rather than only chromosome 19. After checking quality and obtaining a complete gene count table, I would compare SNI-Sal vs Sham-Sal (injury effect), Sham-Oxy vs Sham-Sal (oxycodone effect without injury), and SNI-Oxy vs SNI-Sal (oxycodone effect with injury). I would also test whether the oxycodone effect differs between SNI and Sham mice. In R, I would use DESeq2 for differential expression, ggplot2 for plots, and clusterProfiler for pathway analysis. The two samples analyzed so far are useful for testing the workflow, but one sample per group is insufficient for reliable differential expression testing.